# Lab 09 — Cortex Search Service

A **managed semantic search service** that handles indexing, embedding, and retrieval automatically.

| Item | Detail |
|---|---|
| **Feature** | `CREATE CORTEX SEARCH SERVICE` |
| **Data** | `wiki_articles` table (created in Lab 06) |
| **Exam Domain** | 2.0 Gen AI & LLM Functions |
| **You'll learn** | Search service creation, indexing, TARGET_LAG, querying |

> **Prerequisite:** Run Lab 06 first to create the `wiki_articles` table.


## Step 1 — Environment Setup

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Create a Cortex Search Service

One DDL statement creates a fully managed search index.

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE wiki_search
    ON body
    ATTRIBUTES title, category
    WAREHOUSE = DS_WH
    TARGET_LAG = '1 minute'
    AS (
        SELECT article_id, title, category, body
        FROM wiki_articles
    );

**What just happened:**
- Snowflake embedded all `body` text into vectors automatically
- Created an inverted index for keyword + semantic search
- `TARGET_LAG = '1 minute'` means the index refreshes within 1 minute of source changes
- `ATTRIBUTES` (title, category) enable filtering in search queries


## Step 3 — Manual Vector Search (for comparison)

Before Cortex Search existed, you had to manage embeddings yourself:

In [ ]:
CREATE OR REPLACE TABLE article_embeddings AS
SELECT article_id, title, category, body,
    SNOWFLAKE.CORTEX.EMBED_TEXT_1024('e5-base-v2', body) AS embedding
FROM wiki_articles;

In [ ]:
SELECT title, category,
    ROUND(VECTOR_COSINE_SIMILARITY(
        embedding,
        SNOWFLAKE.CORTEX.EMBED_TEXT_1024('e5-base-v2', 'How do vector databases enable AI applications?')
    ), 4) AS similarity
FROM article_embeddings
ORDER BY similarity DESC
LIMIT 3;

## Key Difference

| Approach | Pros | Cons |
|---|---|---|
| Manual embeddings | Full control, custom models | You manage storage, indexing, refresh |
| Cortex Search Service | Managed, auto-refresh, hybrid search | Less control over embedding model |


## Key Takeaways

- **Cortex Search Service** = managed semantic search. One DDL, no infrastructure.
- `TARGET_LAG` controls index freshness — balance cost vs. freshness.
- Automatically combines keyword + semantic (hybrid) search.
- `ATTRIBUTES` enable server-side filtering in search queries.
- Manual vector search gives more control but requires you to manage everything.
